[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lab-biotek-bio-ugm/BIMB26512_Praktikum_Rekayasa_Biologi/blob/main/notebooks/M03_michaelis_menten.ipynb)

# Bioengineering Practicum · BIMB265125
## Meeting 3 — Python for simulating biological systems: enzyme kinetics

**Universitas Gadjah Mada · Faculty of Biology · Master's Programme in Biology**
Matin Nuhamunada, S.Si., M.Sc., Ph.D. · 25 September 2026

**Reading:** B. P. Ingalls (2013), *Mathematical Modeling in Systems Biology: An Introduction*,
MIT Press — Chapter 3 (Biochemical kinetics: Michaelis–Menten kinetics, regulation of enzyme
activity, cooperativity) and the section on separation of time-scales / model reduction in Chapter 2.
We use Ingalls' notation: lower-case $s, e, c, p$ for concentrations, $e_T$ for total enzyme.

---

### Before you start

1. **File → Save a copy in Drive.** Work in *your* copy.
2. Fill in your name and NIM below.
3. When you are finished: **Runtime → Restart and run all**. If it does not run clean
   from top to bottom, it is not finished.
4. **File → Download → Download .ipynb**, then put it in
   `BIMB265125_<yourname>/M03_enzyme/notebooks/`.

### What you are building on

Last week every model was mass action and written by hand. Today's single example — one enzyme
converting one substrate — is enough to see why biology needs *more* than mass action, and why
we need Python to do the bookkeeping.

| Last week | Today |
|---|---|
| One hand-written `rhs` per model | The mechanism stored as **data**; one generic `rhs` |
| Mass-action rate laws | Deriving the **Michaelis–Menten** rate law by model reduction |
| `solve_ivp` with its defaults | Why fast binding makes a model **stiff**, and which solver to use |
| Plot the simulation | Run a **virtual assay**, fit $V_{max}$ and $K_M$, then add an **inhibitor** |

In [1]:
NAME  = ''      # e.g. 'Siti Rahmawati'
NIM = ''      # e.g. 'BI12345'

assert NAME and NIM, 'Fill in NAME and NIM before you go on.'
print(f'{NAME} — {NIM}')

A — 2


In [2]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit

plt.rcParams['figure.figsize'] = (7, 4)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

print('numpy', np.__version__, '· pandas', pd.__version__)

numpy 2.5.3 · pandas 3.0.6


---
## Task 1 — The mechanism, written as data

The simplest enzyme mechanism (Ingalls §3.1):

$$S + E \;\underset{k_{-1}}{\overset{k_1}{\rightleftharpoons}}\; C \;\xrightarrow{k_2}\; E + P$$

Three elementary reactions, so mass action applies to each one:
$v_1 = k_1 s e$, $\;v_{-1} = k_{-1} c$, $\;v_2 = k_2 c$.

Recall the recipe: species → reactions with rates → bookkeeping. The bookkeeping *is* the
stoichiometric matrix $S$, so every model becomes

$$\frac{d\mathbf{x}}{dt} = S\,\mathbf{v}(\mathbf{x})$$

Below, a network is a **dictionary** holding a list of reactions, and three short functions do
the rest. Read them — you will not need to change them today.

In [3]:
def stoich_matrix(net):
    """Rows = species, columns = reactions."""
    sp = net['species']
    S = np.zeros((len(sp), len(net['reactions'])))
    for j, r in enumerate(net['reactions']):
        for name, coeff in r['stoich'].items():
            S[sp.index(name), j] = coeff
    return S

def make_rhs(net):
    """Turn a network into a function solve_ivp understands."""
    S, sp, k = stoich_matrix(net), net['species'], net['params']
    def rhs(t, y):
        x = dict(zip(sp, y))                               # {'s': 4.2, 'e': 0.1, ...}
        v = np.array([r['rate'](x, k) for r in net['reactions']])
        return S @ v
    return rhs

def simulate(net, y0, t_end, n=400, **solver_options):
    """y0 is a dict {species: value}; species not listed start at 0.
    Returns a DataFrame: one row per time point, one column per species."""
    y = [y0.get(name, 0.0) for name in net['species']]
    # time grid: even spacing plus log spacing, so the fast start is resolved too
    t = np.unique(np.concatenate([np.linspace(0, t_end, n), np.geomspace(t_end * 1e-5, t_end, n)]))
    sol = solve_ivp(make_rhs(net), (0, t_end), y, t_eval=t, **solver_options)
    assert sol.success, sol.message
    df = pd.DataFrame(sol.y.T, index=pd.Index(sol.t, name='t'), columns=net['species'])
    df.attrs['nfev'] = sol.nfev                            # how many times rhs was called
    return df

### Your turn — write the three reactions

Fill in `reactions` with one dictionary per elementary reaction, in the same format as last
week's example: `{'name': ..., 'stoich': {species: coefficient}, 'rate': lambda x, k: ...}`.
Inside a rate law, `x['s']` is the substrate concentration and `k['k1']` a rate constant.

In [4]:
enzyme = {
    'species': ['s', 'e', 'c', 'p'],
    'params':  {'k1': 30.0, 'km1': 1.0, 'k2': 10.0},     # 1/(mM·s), 1/s, 1/s
    'reactions': [
        # TODO  s + e -> c      rate k1 * s * e
        # TODO  c -> s + e      rate km1 * c
        # TODO  c -> e + p      rate k2 * c
    ],
}

In [5]:
# --- the check ---------------------------------------------------------
assert len(enzyme['reactions']) == 3, 'Write all three reactions.'
S = stoich_matrix(enzyme)
display(pd.DataFrame(S, index=enzyme['species'], columns=[r['name'] for r in enzyme['reactions']]))

# conservation laws live in S: if w @ S == 0 then w . x never changes
for label, w in {'total enzyme      e + c':     [0, 1, 1, 0],
                 'total substrate   s + c + p': [1, 0, 1, 1]}.items():
    print(f'{label}:  w @ S = {np.array(w) @ S}')
    assert np.allclose(np.array(w) @ S, 0), f'{label} is not conserved - check your stoichiometry.'

eT, s0 = 1.0, 5.0
full = simulate(enzyme, {'s': s0, 'e': eT}, t_end=2, rtol=1e-10, atol=1e-12)
assert np.allclose(full.e + full.c, eT) and np.allclose(full.s + full.c + full.p, s0)
assert full.p.iloc[-1] > 0.99 * s0, 'The substrate should be (almost) all converted by t = 2.'
print('PASS')

ax = full.plot(lw=2.2, title=f'Full mass-action model, e_T = {eT}, s0 = {s0}')
ax.set_xlabel('time (s)'); ax.set_ylabel('concentration (mM)')
ax.figure.savefig('M03_fig01_full_model.png', dpi=150, bbox_inches='tight')
plt.show()

AssertionError: Write all three reactions.

**Q1.1** Four species but only **two** independent variables. Use the two conservation laws to
say why.
**Q1.2** Look at $c(t)$. It rises within a fraction of a second and then changes slowly. Which
rate constant(s) set the speed of that first rise?

*Your answer:*

---
## Task 2 — Model reduction: where Michaelis–Menten comes from

The complex $c$ settles almost immediately and then *tracks* the slowly falling substrate. The
**quasi-steady-state approximation** (Briggs & Haldane, 1925) says: treat $c$ as if it were always at
steady state, $dc/dt \approx 0$. With $e = e_T - c$:

$$0 = k_1 (e_T - c)s - (k_{-1} + k_2)c \quad\Longrightarrow\quad c = \frac{e_T\, s}{K_M + s},
\qquad K_M = \frac{k_{-1} + k_2}{k_1}$$

and the rate of product formation becomes the Michaelis–Menten rate law

$$v = k_2 c = \frac{V_{max}\, s}{K_M + s}, \qquad V_{max} = k_2 e_T$$

Four equations and three parameters became **one** equation and two parameters. Is it any good?
The reduced model is just another network — one reaction with a non-mass-action rate law.

In [ ]:
k = enzyme['params']
KM   = (k['km1'] + k['k2']) / k['k1']
Vmax = k['k2'] * eT
print(f'K_M = {KM:.4f} mM    V_max = {Vmax:.2f} mM/s')

mm = {
    'species': ['s', 'p'],
    'params':  {'Vmax': Vmax, 'KM': KM},
    'reactions': [
        {'name': 's -> p (MM)', 'stoich': {'s': -1, 'p': +1},
         'rate': lambda x, k: k['Vmax'] * x['s'] / (k['KM'] + x['s'])},
    ],
}
red = simulate(mm, {'s': s0}, t_end=2, rtol=1e-10, atol=1e-12)
c_qss = eT * full.s / (KM + full.s)                     # what the QSSA says c should be

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
f, r = full.iloc[1:], red.iloc[1:]                      # drop t = 0 for the log axis
ax1.semilogx(f.index, f.s, 'C0', lw=2.2, label='s  full')
ax1.semilogx(r.index, r.s, 'C0--', lw=2, label='s  Michaelis–Menten')
ax1.semilogx(f.index, f.p, 'C1', lw=2.2, label='p  full')
ax1.semilogx(r.index, r.p, 'C1--', lw=2, label='p  Michaelis–Menten')
ax1.semilogx(f.index, f.c, 'C2', lw=2.2, label='c  full')
ax1.semilogx(f.index, c_qss.iloc[1:], 'C2:', lw=2, label='c  quasi-steady state')
ax1.set_xlabel('time (s, log scale)'); ax1.set_ylabel('concentration (mM)'); ax1.legend(fontsize=8)

# phase plane (s, c): eliminate e = eT - c, so the full model is 2-D
S_g, C_g = np.meshgrid(np.linspace(0, s0, 25), np.linspace(0, eT, 25))
dS = -k['k1'] * (eT - C_g) * S_g + k['km1'] * C_g
dC =  k['k1'] * (eT - C_g) * S_g - (k['km1'] + k['k2']) * C_g
ax2.streamplot(S_g, C_g, dS, dC, color='lightgrey', density=1.2)
s_line = np.linspace(0, s0, 200)
ax2.plot(s_line, eT * s_line / (KM + s_line), 'C2:', lw=2.5, label='dc/dt = 0 (QSSA curve)')
ax2.plot(full.s, full.c, 'k', lw=2, label='full model trajectory')
ax2.plot(s0, 0, 'ko')
ax2.set_xlabel('substrate s (mM)'); ax2.set_ylabel('complex c (mM)'); ax2.legend(fontsize=8, loc='lower right')
fig.savefig('M03_fig02_qssa.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'fast time-scale  1/(k1 (s0 + KM)) = {1 / (k["k1"] * (s0 + KM)):.4f} s')
print(f'slow time-scale  (KM + s0)/Vmax   = {(KM + s0) / Vmax:.4f} s')

**Q2.1** In the phase plane, describe the trajectory in two phases: what happens first, what
happens after? Relate each phase to one of the two printed time-scales.
**Q2.2** Michaelis & Menten's original argument (1913) was different: *rapid equilibrium*,
assuming binding is much faster than catalysis ($k_{-1} \gg k_2$), which gives $K_M = k_{-1}/k_1$.
Compute that value for our parameters. Would it be a good approximation here? Why not?

*Your answer:*

---
## Task 3 — When is the approximation allowed? (and why the solver cares)

The QSSA needs $c$ to be fast compared with $s$. A standard criterion (Segel, 1988) is

$$\frac{e_T}{K_M + s_0} \ll 1$$

— little enzyme compared with substrate, which is the normal situation in a test tube but
*not* always in a cell. Below we scan $e_T$ over three orders of magnitude, record the error of the
reduced model, and at the same time record how hard the solvers had to work. Each run is one
**row** (a dictionary); the rows become a `DataFrame`.

In [ ]:
rows = []
for eT_i in np.geomspace(1e-3, 3, 8):
    t_end = 3 * (KM + s0) / (k['k2'] * eT_i)           # ~3 slow time-scales: substrate used up
    y0 = {'s': s0, 'e': eT_i}
    opts = dict(rtol=1e-8, atol=1e-12)
    ref = simulate(enzyme, y0, t_end, method='LSODA', **opts)
    t0 = time.perf_counter()
    rk = simulate(enzyme, y0, t_end, method='RK45', **opts)
    rk_seconds = time.perf_counter() - t0
    red_i = simulate({**mm, 'params': {'Vmax': k['k2'] * eT_i, 'KM': KM}}, {'s': s0}, t_end, method='LSODA', **opts)
    rows.append({
        'eT': eT_i,
        'eT/(KM+s0)': eT_i / (KM + s0),
        'MM error (max |Δp| / s0)': (ref.p - red_i.p).abs().max() / s0,
        'RK45 rhs calls': rk.attrs['nfev'],
        'LSODA rhs calls': ref.attrs['nfev'],
        'RK45 seconds': round(rk_seconds, 2),
    })
scan = pd.DataFrame(rows)
scan

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.loglog(scan['eT/(KM+s0)'], scan['MM error (max |Δp| / s0)'], 'o-')
ax1.axhline(0.01, color='grey', ls=':', label='1% error')
ax1.set_xlabel('e_T / (K_M + s0)'); ax1.set_ylabel('error of the Michaelis–Menten model'); ax1.legend()

ax2.loglog(scan['eT/(KM+s0)'], scan['RK45 rhs calls'], 'o-', label='RK45 (explicit, the default)')
ax2.loglog(scan['eT/(KM+s0)'], scan['LSODA rhs calls'], 's-', label='LSODA (switches to implicit)')
ax2.set_xlabel('e_T / (K_M + s0)'); ax2.set_ylabel('rhs calls for the full model'); ax2.legend()
fig.savefig('M03_fig03_validity_and_stiffness.png', dpi=150, bbox_inches='tight')
plt.show()

**Q3.1** Use pandas to find the largest $e_T$ for which the error stays below 1%
(hint: `scan[scan['MM error (max |Δp| / s0)'] < 0.01]`). Does that agree with Segel's criterion?
**Q3.2** The Michaelis–Menten approximation gets *better* exactly where `RK45` gets *slower*.
Explain why these are the same fact. (Hint: a **stiff** system is one with widely separated time-scales.)
**Q3.3** You want to simulate a metabolic pathway with 20 enzymes. Give two reasons to use
Michaelis–Menten rate laws instead of the full mechanisms, and one reason not to.

*Your answer:*

In [ ]:
# Q3.1 - your code here

---
## Task 4 — The virtual lab: measuring $V_{max}$ and $K_M$

In the lab you never see $k_1$, $k_{-1}$, $k_2$. You mix enzyme with different starting substrate
concentrations, measure the **initial rate** $v_0$ (the steepest slope of the product curve), and
fit $V_{max}$ and $K_M$. Here the "lab" is the full model plus 5% measurement noise — so you know
the true answer and can check your method.

In [ ]:
eT_assay = 0.01                                          # assays use little enzyme (Task 3!)
s0_series = np.array([0.05, 0.1, 0.2, 0.4, 0.8, 1.5, 3.0, 6.0])   # mM

def initial_rate(net, y0, t_end=5):
    """Steepest slope of the product curve - what you would read off a progress curve."""
    df = simulate(net, y0, t_end, method='LSODA', rtol=1e-8, atol=1e-12)
    return np.gradient(df.p.to_numpy(), df.index.to_numpy()).max()

rng = np.random.default_rng(265125)
assay = pd.DataFrame({'s0': s0_series})
assay['v0'] = [initial_rate(enzyme, {'s': s, 'e': eT_assay}) for s in assay.s0]
assay['v0_measured'] = assay.v0 * (1 + 0.05 * rng.standard_normal(len(assay)))
assay

In [ ]:
def michaelis_menten(s, Vmax, KM):
    return Vmax * s / (KM + s)

# method 1: nonlinear least squares directly on v0 vs s0
(Vmax_nl, KM_nl), _ = curve_fit(michaelis_menten, assay.s0, assay.v0_measured,
                                p0=[assay.v0_measured.max(), assay.s0.median()])

# method 2: Lineweaver-Burk - a straight line through 1/v0 vs 1/s0
slope, intercept = np.polyfit(1 / assay.s0, 1 / assay.v0_measured, 1)
Vmax_lb, KM_lb = 1 / intercept, slope / intercept

results = pd.DataFrame({
    'truth':               {'Vmax': k['k2'] * eT_assay, 'KM': KM},
    'nonlinear fit':       {'Vmax': Vmax_nl, 'KM': KM_nl},
    'Lineweaver–Burk':     {'Vmax': Vmax_lb, 'KM': KM_lb},
})
display(results.round(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
s_line = np.linspace(0, 6.5, 300)
ax1.plot(assay.s0, assay.v0_measured, 'ko', label='virtual assay')
ax1.plot(s_line, michaelis_menten(s_line, Vmax_nl, KM_nl), label='nonlinear fit')
ax1.plot(s_line, michaelis_menten(s_line, Vmax_lb, KM_lb), '--', label='Lineweaver–Burk')
ax1.set_xlabel('s0 (mM)'); ax1.set_ylabel('initial rate v0 (mM/s)'); ax1.legend()

x_line = np.linspace(0, 1 / assay.s0.min(), 100)
ax2.plot(1 / assay.s0, 1 / assay.v0_measured, 'ko')
ax2.plot(x_line, slope * x_line + intercept, '--', color='C1')
ax2.set_xlabel('1 / s0'); ax2.set_ylabel('1 / v0'); ax2.set_title('Lineweaver–Burk plot')
fig.savefig('M03_fig04_virtual_assay.png', dpi=150, bbox_inches='tight')
plt.show()

**Q4.1** Which method recovered the true values better? Look at where the points sit on the
Lineweaver–Burk plot: which measurements end up with the most leverage on the line, and are
those the precise ones or the noisy ones?
**Q4.2** Change the seed of `rng` (or the noise to 10%) and rerun both cells two or three times.
Which method's estimates jump around more?
**Q4.3** Why must the assay use $e_T$ = 0.01 and not $e_T$ = 1? Answer using your Task 3 result.

*Your answer:*

---
## Task 5 — Regulation: a competitive inhibitor

A competitive inhibitor $I$ binds the free enzyme at the active site, so enzyme bound to $I$
cannot bind substrate (Ingalls §3.2):

$$E + I \;\underset{k_{-3}}{\overset{k_3}{\rightleftharpoons}}\; C_I \qquad K_i = \frac{k_{-3}}{k_3}$$

The QSSA then predicts that $V_{max}$ is unchanged but the **apparent** $K_M$ grows:

$$v = \frac{V_{max}\, s}{K_M\left(1 + \dfrac{i}{K_i}\right) + s}$$

Here the "model as data" pays off: the inhibited network is the old reaction list **plus two
entries**. No function changes.

In [ ]:
inhibited = {
    'species': enzyme['species'] + ['i', 'ci'],
    'params':  {**enzyme['params'], 'k3': 10.0, 'km3': 5.0},   # Ki = km3/k3 = 0.5 mM
    'reactions': enzyme['reactions'] + [
        # TODO  e + i -> ci      rate k3 * e * i
        # TODO  ci -> e + i      rate km3 * ci
    ],
}

In [ ]:
# --- the check ---------------------------------------------------------
assert len(inhibited['reactions']) == 5, 'Add both inhibitor reactions.'
Si = stoich_matrix(inhibited)
for label, w in {'total enzyme     e + c + ci': [0, 1, 1, 0, 0, 1],
                 'total inhibitor  i + ci':     [0, 0, 0, 0, 1, 1]}.items():
    assert np.allclose(np.array(w) @ Si, 0), f'{label} is not conserved - check your stoichiometry.'
print('PASS')

iT = 1.0
Ki = inhibited['params']['km3'] / inhibited['params']['k3']
# pre-incubate: enzyme and inhibitor already at binding equilibrium when substrate is added
e_free = eT_assay / (1 + iT / Ki)
y0_inh = {'e': e_free, 'ci': eT_assay - e_free, 'i': iT - (eT_assay - e_free)}

assay['v0_inhibited'] = [initial_rate(inhibited, {**y0_inh, 's': s}) for s in assay.s0]

fits = {}
for col in ['v0', 'v0_inhibited']:
    fits[col], _ = curve_fit(michaelis_menten, assay.s0, assay[col], p0=[assay[col].max(), 1.0])
comparison = pd.DataFrame(fits, index=['Vmax', 'KM (apparent)']).T
comparison.loc['QSSA prediction, inhibited'] = [k['k2'] * eT_assay, KM * (1 + iT / Ki)]
display(comparison.round(4))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for col, label in [('v0', 'no inhibitor'), ('v0_inhibited', f'i_T = {iT} mM')]:
    Vm, Km = fits[col]
    ax1.plot(assay.s0, assay[col], 'o', label=label)
    ax1.plot(s_line, michaelis_menten(s_line, Vm, Km), color=ax1.lines[-1].get_color())
    x = np.linspace(-1 / Km, 1 / assay.s0.min(), 100)
    ax2.plot(1 / assay.s0, 1 / assay[col], 'o')
    ax2.plot(x, (Km / Vm) * x + 1 / Vm, color=ax2.lines[-1].get_color(), label=label)
ax1.set_xlabel('s0 (mM)'); ax1.set_ylabel('v0 (mM/s)'); ax1.legend()
ax2.axvline(0, color='k', lw=0.8); ax2.axhline(0, color='k', lw=0.8)
ax2.set_xlabel('1 / s0'); ax2.set_ylabel('1 / v0'); ax2.set_title('Lineweaver–Burk'); ax2.legend()
fig.savefig('M03_fig05_inhibition.png', dpi=150, bbox_inches='tight')
plt.show()

**Q5.1** Does the simulation confirm the QSSA prediction for $V_{max}$ and the apparent $K_M$?
Explain in molecular terms why enough substrate can always out-compete a competitive inhibitor.
**Q5.2** Where do the two Lineweaver–Burk lines cross? That crossing point is the diagnostic
"fingerprint" of competitive inhibition.
**Q5.3** A *non-competitive* inhibitor can bind both $E$ and $C$, and $C_I$-bound complex makes no
product. Which reactions would you add to the list? Predict what happens to $V_{max}$ and $K_M$.

*Your answer:*

---
## Optional — cooperativity and the Hill function

Many regulatory proteins bind several ligands, and binding of one makes the next easier
(Ingalls §3.3). The result is a *sigmoidal* response, summarised by the Hill function

$$v = \frac{V_{max}\, s^n}{K^n + s^n}$$

with Michaelis–Menten as the special case $n = 1$. Implement the Q5.3 network above if you
have not yet, or explore the Hill function below: how many-fold must $s$ increase to take the
response from 10% to 90% of $V_{max}$, for $n = 1, 2, 4$? Why would a cell want a large $n$ for a switch?

In [ ]:
s_h = np.geomspace(0.01, 100, 400)
fig, ax = plt.subplots()
for n in (1, 2, 4):
    ax.semilogx(s_h, s_h**n / (1 + s_h**n), lw=2, label=f'n = {n}')
ax.set_xlabel('s / K'); ax.set_ylabel('v / Vmax'); ax.legend(); plt.show()

# your code here

---
## Checklist before you submit

- [ ] `NAME` and `NIM` are filled in
- [ ] The `enzyme` and `inhibited` networks both pass their checks
- [ ] **Runtime → Restart and run all** completes with no errors
- [ ] Five figures saved: `M03_fig01` … `M03_fig05`
- [ ] Every *Your answer* cell is filled in
- [ ] Downloaded as `.ipynb` into `M03_enzyme/notebooks/`
- [ ] Export as PDF
---

**Next week — Meeting 4:** Flux Balance Analysis with COBRApy. Keep the stoichiometric matrix
from Task 1, throw away the rate laws — you have just seen how hard they are to measure — and
ask what $S\mathbf{v} = 0$ alone can tell you.